# Day 33：高级RAG技术(下)：BM25混合检索、RRF融合、bge-reranker重排与父文档检索（Jupyter Notebook 实战手册）

> **所属阶段**：第 3 阶段 · 第三阶段：LangChain 与企业级 RAG 开发 ｜ **第 5 周**：第5周：企业级 RAG 全链路、高级检索优化与评估  
> **GitLab 工单**：`STARMIND-033` ｜ **开发分支**：`feat/day33-hybrid-rerank-parent-doc`  
> **配套 HTML5 课件**：`courseware/day33/index.html` ｜ **独立版本快照**：`versions/day33/`

---

## 🎙️ 老陈开场旁白

早啊各位！昨天留下的悬念今天直接揭晓：为什么大厂生产环境几乎**没有一家只用纯向量检索**？因为纯向量（Dense）擅长语义泛化，却天生瞎眼——你搜 `ERR-5042`、`STARMIND-033` 或者内部代号「天枢项目」，它在 Embedding 空间里就是一堆生僻字符，得分惨不忍睹！
今天老陈带大家把高级 RAG 的后半套压箱底绝活全配齐，组成真正的**大厂标配工业级检索架构**：
1. **BM25 + 向量混合检索（Hybrid Search）**：字面精确匹配 + 语义泛化两条腿走路；
2. **RRF（倒数排名融合）**：无须调参，优雅解决 BM25 分数无界与余弦相似度 `[0,1]` 量纲不统一的融合难题；
3. **Cross-Encoder 重排序（bge-reranker 原理）**：粗排拉回 Top-10，再让交叉编码器把 `(Query, Doc)` 拼在一起做逐词注意力精排，把最准的文档死死按在 Top-1；
4. **父文档检索器（ParentDocumentRetriever）**：用小切片精准检索、用完整父文档喂给大模型，鱼和熊掌兼得！

### 承上启下
- **昨日基线**：昨天（Day 32）我们用 Query Rewriting、Multi-Query、HyDE 和上下文压缩解决了「员工口语黑话」的表达鸿沟。但测试同学一测研发故障库又报了新 Bug：员工直接搜「ERR-5042 报错怎么排查」或「天枢项目审批流程」，纯向量检索居然把完全不沾边的文档排在前面——因为向量模型压根不认识公司内部代号和错误码！
- **明日预告**：明天（Day 34）我们将从「调优」跨入「科学评测」：构建企业级 RAG 自动化评测框架（Faithfulness、答案相关性、上下文精确率/召回率），把 RAG 质量变成 CI 流水线里的硬门禁！


In [ ]:
# 环境自检与路径初始化：确保可以无缝导入当日版本模块并支持异步协程运行
import os
import sys
from pathlib import Path

# 允许在 Jupyter 已有事件循环内安全调用 asyncio.run()
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'versions').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DAY_DIR = PROJECT_ROOT / 'versions' / 'day33'
for _p in (DAY_DIR, DAY_DIR / 'src', PROJECT_ROOT):
    if _p.exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
os.chdir(DAY_DIR)
__file__ = str(DAY_DIR / 'src' / 'notebook_entry.py')

print(f'✅ Python 版本: {sys.version.split()[0]}')
print(f'✅ 项目根目录: {PROJECT_ROOT}')
print(f'✅ 当日版本目录: {DAY_DIR}')


## 📋 企业需求背景与验收目标（STARMIND-033）

星辰智脑知识库即将接入研发运维故障排查手册（含大量错误码如 ERR-5042、接口名、项目代号如「天枢项目」）与长篇制度全文。纯向量检索在精确编码匹配与长文档上下文完整性上暴露严重短板。

**今日迭代目标：**
- 实现支持中文二元词元 + 英文/错误码精确切分的 `BM25Index` 与 `DenseVectorIndex`，并基于 RRF 实现 `HybridRRFRetriever`。
- 实现类 `bge-reranker` 的交叉交互重排器 `BGERerankerSimulator`，完成两阶段「粗排 -> 精排」流水线。
- 实现 `ParentDocumentRetriever`：子切片检索 + 按 `parent_id` 去重回溯完整父文档。

**验收标准：**
- 针对「ERR-5042 排查」「天枢项目审批」等精确实体查询，Hybrid RRF 的 Top-1 命中率达到 100%。
- Reranker 精排后，混淆候选文档中的核心答案切片排位显著前移。
- ParentDocumentRetriever 检索命中同一父文档多个子块时能自动去重，并完整返回父文档正文。
- 所有 pytest 自动化单元测试 100% 本地离线通过。

---

## 📚 课堂核心技术笔记

### 1. 1. Okapi BM25 为什么吊打传统 TF-IDF？

BM25 在 TF-IDF 基础上引入了两个极具工程智慧的改进：
- **词频饱和度参数 `k1`（默认 1.5）**：某词在文档里出现 1 次到 5 次相关性上升明显，但出现 50 次并不比 10 次更相关（防止关键词堆砌作弊）。
- **文档长度归一化参数 `b`（默认 0.75）**：一篇 5000 字长文里偶尔提到 1 次「VPN」，其含金量远不如 100 字短文里提到 1 次「VPN」，BM25 通过 `dl / avgdl` 自动对过长文档施加惩罚。

> **⚠️ 老陈避坑**：踩坑提醒：中文 BM25 千万不要只按空格 `split()` 分词！否则整句中文会变成一个巨型 Token 导致 BM25 完全失效。离线轻量场景可用「英文/数字整词 + 中文单字与二元字（Bigram）」，生产环境推荐 `jieba` + 企业自定义专有词典（加入项目代号、错误码）。

### 2. 2. 生产环境接入 BAAI/bge-reranker-v2-m3 标准写法

国内企业最常用的开源多语言精排模型是智源研究院的 `BAAI/bge-reranker-v2-m3`（或通过 SiliconFlow / Cohere / DashScope Rerank API 调用）：
```python
from FlagEmbedding import FlagReranker

reranker = FlagReranker('BAAI/bge-reranker-v2-m3', use_fp16=True)
pairs = [[query, doc.page_content] for doc in candidate_docs]
scores = reranker.compute_score(pairs, normalize=True)  # 输出 0~1 概率值
```

> **⚠️ 老陈避坑**：踩坑提醒：Reranker 的输入候选数不要贪多！粗排给 Reranker 喂 10~20 条文档最划算，如果喂 100 条文档，Cross-Encoder 推理延迟会直接突破 1 秒，得不偿失。


## 💻 实操步骤 1：步骤一：实现中英混合分词器、Okapi BM25、Dense 向量索引与 RRF 混合检索器

- **目标文件**：`versions/day33/src/day33_hybrid_rrf.py`
- **老陈代码旁白**：咱们完全不依赖外部 API，先用纯 Python 手写一个工业级标准的 `BM25Index`、`DenseVectorIndex` 和 `HybridRRFRetriever`。重点看分词器是如何同时保留 `ERR-5042` 这种错误码和中文二元词组的！


In [ ]:
"""
Day 33 - 步骤一：BM25 稀疏检索 + Dense 向量检索 + RRF 倒数排名融合
对应生产栈：rank_bm25 / Elasticsearch BM25 + Milvus/Qdrant + EnsembleRetriever(RRF)
"""

from __future__ import annotations

import hashlib
import math
import re
from collections import Counter
from dataclasses import dataclass, field
from typing import Any, Dict, List, Sequence, Tuple


@dataclass
class Document:
    """统一文档切片模型（与 LangChain Document 接口对齐）。"""

    page_content: str
    metadata: Dict[str, Any] = field(default_factory=dict)

    @property
    def doc_id(self) -> str:
        if "doc_id" in self.metadata:
            return str(self.metadata["doc_id"])
        return hashlib.md5(self.page_content.encode("utf-8")).hexdigest()[:12]


def tokenize_mixed_cn_en(text: str) -> List[str]:
    """
    中英混合 + 错误码友好分词器：
    1. 完整提取英文词、错误码与编号（如 ERR-5042、STARMIND-033、VPN、OA）；
    2. 对中文连续片段同时提取单字与二元字（Bigram），兼顾召回与词组精度。
    """
    lowered = text.lower()
    tokens: List[str] = []

    # 提取英文/数字/连字符组合（完整保留错误码与工单号）
    alnum_tokens = re.findall(r"[a-z0-9]+(?:[-_][a-z0-9]+)*", lowered)
    tokens.extend(alnum_tokens)

    # 提取中文连续串并生成单字 + 二元词组
    cn_chunks = re.findall(r"[\u4e00-\u9fff]+", lowered)
    for chunk in cn_chunks:
        for ch in chunk:
            tokens.append(ch)
        for i in range(len(chunk) - 1):
            tokens.append(chunk[i : i + 2])
    return tokens


class BM25Index:
    """
    标准 Okapi BM25 稀疏检索引擎：
    score(q, d) = sum_{t in q} IDF(t) * (f(t,d) * (k1 + 1)) / (f(t,d) + k1 * (1 - b + b * |d| / avgdl))
    """

    def __init__(self, k1: float = 1.5, b: float = 0.75) -> None:
        self.k1 = k1
        self.b = b
        self.documents: List[Document] = []
        self.doc_term_freqs: List[Counter[str]] = []
        self.doc_lengths: List[int] = []
        self.df: Counter[str] = Counter()
        self.avgdl: float = 0.0

    def add_documents(self, docs: Sequence[Document]) -> None:
        for doc in docs:
            tokens = tokenize_mixed_cn_en(doc.page_content)
            tf = Counter(tokens)
            self.documents.append(doc)
            self.doc_term_freqs.append(tf)
            self.doc_lengths.append(len(tokens))
            for term in tf.keys():
                self.df[term] += 1
        total_len = sum(self.doc_lengths)
        self.avgdl = total_len / max(1, len(self.documents))

    def _idf(self, term: str) -> float:
        n = len(self.documents)
        df = self.df.get(term, 0)
        return math.log(1.0 + (n - df + 0.5) / (df + 0.5))

    def search(self, query: str, k: int = 5) -> List[Tuple[Document, float]]:
        q_tokens = tokenize_mixed_cn_en(query)
        if not q_tokens or not self.documents:
            return []

        scored: List[Tuple[Document, float]] = []
        for idx, doc in enumerate(self.documents):
            tf_map = self.doc_term_freqs[idx]
            dl = self.doc_lengths[idx]
            score = 0.0
            for term in q_tokens:
                freq = tf_map.get(term, 0)
                if freq == 0:
                    continue
                idf = self._idf(term)
                denom = freq + self.k1 * (1.0 - self.b + self.b * dl / max(1e-9, self.avgdl))
                score += idf * (freq * (self.k1 + 1.0)) / denom
            if score > 0:
                scored.append((doc, round(score, 4)))

        scored.sort(key=lambda x: x[1], reverse=True)
        return scored[:k]


class DenseVectorIndex:
    """
    轻量确定性语义向量索引（离线模拟 bge-m3 语义检索能力）：
    内置企业领域同义词语义桶 + 特征哈希投影，支持余弦相似度检索。
    """

    SEMANTIC_GROUPS: Dict[str, Tuple[str, ...]] = {
        "travel_hotel": ("出差", "差旅", "酒店", "住宿", "住店", "魔都", "上海", "北京", "一线城市", "报销", "上限"),
        "gateway_timeout": ("网关", "超时", "504", "熔断", "连接池", "上游", "延迟", "故障", "排查"),
        "deploy_window": ("发布", "上线", "发版", "窗口", "周五", "合并", "变更", "生产"),
        "vpn_security": ("远程", "居家", "内网", "令牌", "零信任", "vpn", "安全", "脱敏"),
        "project_approval": ("审批", "立项", "预算", "负责人", "流程", "项目"),
        "leave_policy": ("年假", "病假", "休假", "请假", "诊断证明", "工龄"),
    }

    def __init__(self, dim: int = 96) -> None:
        self.dim = dim
        self.documents: List[Document] = []
        self.vectors: List[List[float]] = []

    def embed(self, text: str) -> List[float]:
        vec = [0.0] * self.dim
        lowered = text.lower()

        # 1. 前 12 维编码企业语义概念桶（模拟语义泛化能力）
        for g_idx, words in enumerate(self.SEMANTIC_GROUPS.values()):
            hits = sum(1 for w in words if w in lowered)
            if hits > 0:
                vec[g_idx] += 2.5 * math.log1p(hits)

        # 2. 其余维度采用中文二元词特征哈希（有意淡化纯错误码权重，体现语义模型特点）
        cn_chunks = re.findall(r"[\u4e00-\u9fff]+", lowered)
        for chunk in cn_chunks:
            for i in range(len(chunk) - 1):
                bg = chunk[i : i + 2]
                h = int(hashlib.md5(bg.encode("utf-8")).hexdigest(), 16)
                slot = 12 + (h % (self.dim - 12))
                vec[slot] += 1.0

        norm = math.sqrt(sum(v * v for v in vec)) or 1.0
        return [v / norm for v in vec]

    @staticmethod
    def cosine(a: Sequence[float], b: Sequence[float]) -> float:
        return sum(x * y for x, y in zip(a, b))

    def add_documents(self, docs: Sequence[Document]) -> None:
        for doc in docs:
            self.documents.append(doc)
            self.vectors.append(self.embed(doc.page_content))

    def search(self, query: str, k: int = 5) -> List[Tuple[Document, float]]:
        if not self.documents:
            return []
        q_vec = self.embed(query)
        scored = [
            (doc, round(self.cosine(q_vec, d_vec), 4))
            for doc, d_vec in zip(self.documents, self.vectors)
        ]
        scored.sort(key=lambda x: x[1], reverse=True)
        return scored[:k]


class HybridRRFRetriever:
    """
    BM25 + Dense 向量混合检索器（基于 RRF 倒数排名融合）。
    公式：RRF(d) = w_bm25 / (rrf_k + rank_bm25(d)) + w_dense / (rrf_k + rank_dense(d))
    """

    def __init__(
        self,
        bm25_index: BM25Index,
        dense_index: DenseVectorIndex,
        rrf_k: int = 60,
        bm25_weight: float = 0.5,
        dense_weight: float = 0.5,
    ) -> None:
        self.bm25_index = bm25_index
        self.dense_index = dense_index
        self.rrf_k = rrf_k
        self.bm25_weight = bm25_weight
        self.dense_weight = dense_weight

    def retrieve(self, query: str, top_k: int = 5, recall_k: int = 10) -> List[Tuple[Document, float]]:
        bm25_hits = self.bm25_index.search(query, k=recall_k)
        dense_hits = self.dense_index.search(query, k=recall_k)

        rrf_scores: Dict[str, float] = {}
        doc_map: Dict[str, Document] = {}

        for rank, (doc, _score) in enumerate(bm25_hits, start=1):
            did = doc.doc_id
            doc_map[did] = doc
            rrf_scores[did] = rrf_scores.get(did, 0.0) + self.bm25_weight / (self.rrf_k + rank)

        for rank, (doc, _score) in enumerate(dense_hits, start=1):
            did = doc.doc_id
            doc_map[did] = doc
            rrf_scores[did] = rrf_scores.get(did, 0.0) + self.dense_weight / (self.rrf_k + rank)

        fused = [(doc_map[did], round(score, 6)) for did, score in rrf_scores.items()]
        fused.sort(key=lambda x: x[1], reverse=True)
        return fused[:top_k]


def build_enterprise_sample_docs() -> List[Document]:
    """构建星辰数科故障排查 + 制度 + 内部代号混合测试语料。"""
    return [
        Document(
            "【运维故障手册】错误码 ERR-5042 表示星辰统一网关调用上游推理服务超时（默认阈值 15s）。"
            "排查步骤：1. 检查上游 Pod CPU/显存水位；2. 核对 Nginx proxy_read_timeout 配置；3. 开启降级熔断。",
            {"doc_id": "ops-5042", "source": "运维故障排查手册.md", "section": "ERR-5042网关超时"},
        ),
        Document(
            "【运维故障手册】错误码 ERR-4019 表示零信任 VPN 动态令牌校验失败，常见原因是客户端系统时钟偏差超过 30 秒，请同步 NTP 时间。",
            {"doc_id": "ops-4019", "source": "运维故障排查手册.md", "section": "ERR-4019令牌失效"},
        ),
        Document(
            "【项目规范】内部代号「天枢项目」为星辰智脑 2026 核心知识库中台，项目负责人为架构师老陈，预算审批需经 CTO 终审。",
            {"doc_id": "proj-tianshu", "source": "内部项目立项白皮书.md", "section": "天枢项目立项"},
        ),
        Document(
            "【财务报销制度】一线城市（北京、上海、深圳、广州）出差酒店住宿报销上限为每晚 600 元，需提前在 OA 提交差旅申请。",
            {"doc_id": "fin-hotel", "source": "财务报销制度_v3.md", "section": "差旅住宿标准"},
        ),
        Document(
            "【研发规范】生产环境日常发布窗口为周一至周四 10:00-18:00，严禁周五 17:00 之后提交非紧急上线变更。",
            {"doc_id": "dev-window", "source": "研发工程规范.md", "section": "生产发布窗口"},
        ),
    ]


if __name__ == "__main__":
    docs = build_enterprise_sample_docs()
    bm25 = BM25Index()
    dense = DenseVectorIndex()
    bm25.add_documents(docs)
    dense.add_documents(docs)
    hybrid = HybridRRFRetriever(bm25, dense)

    for q in ("ERR-5042 报错怎么排查？", "天枢项目预算找谁审批？", "去魔都出差住店报销上限"):
        print(f"\nQuery: {q}")
        print("  BM25  Top1:", [(d.doc_id, s) for d, s in bm25.search(q, k=1)])
        print("  Dense Top1:", [(d.doc_id, s) for d, s in dense.search(q, k=1)])
        print("  RRF   Top2:", [(d.doc_id, s) for d, s in hybrid.retrieve(q, top_k=2)])


### 🔍 步骤 1 代码深度解析

1. `tokenize_mixed_cn_en` 使用正则 `[a-z0-9]+(?:[-_][a-z0-9]+)*` 完整保护了 `ERR-5042` 这类带连字符的错误码，不会把它拆成碎片。
2. `HybridRRFRetriever` 用 `w / (60 + rank)` 融合 BM25 和 Dense 两路排名，彻底规避了两种打分量纲不一致的问题，在错误码查询（「ERR-5042」）、专有代号（「天枢项目」）和口语语义查询（「魔都住店」）三种截然不同的场景下全部稳居 Top-1！

**预期运行输出参考：**
```text
Query: ERR-5042 报错怎么排查？
  BM25  Top1: [('ops-5042', 5.8214)]
  Dense Top1: [('ops-5042', 0.8944)]
  RRF   Top2: [('ops-5042', 0.016393), ('ops-4019', 0.008065)]

Query: 天枢项目预算找谁审批？
  BM25  Top1: [('proj-tianshu', 6.412)]
  Dense Top1: [('proj-tianshu', 0.8812)]
  RRF   Top2: [('proj-tianshu', 0.016393), ('fin-hotel', 0.008065)]
```


## 💻 实操步骤 2：步骤二：实现 Cross-Encoder 重排器（bge-reranker 原理）与 ParentDocumentRetriever

- **目标文件**：`versions/day33/src/day33_rerank_parent.py`
- **老陈代码旁白**：粗排已经很稳了，现在给它装上两颗涡轮增压引擎：一个是模拟 `bge-reranker-v2-m3` 的交叉编码重排器 `BGERerankerSimulator`，另一个是「小块检索、大块回溯」的 `ParentDocumentRetriever`。


In [ ]:
"""
Day 33 - 步骤二：Cross-Encoder 重排序器（bge-reranker 原理）与父文档检索器（ParentDocumentRetriever）
"""

from __future__ import annotations

import math
import re
import sys
from pathlib import Path
from typing import Dict, List, Sequence, Tuple

ROOT = Path(__file__).resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.day33_hybrid_rrf import (
    BM25Index,
    DenseVectorIndex,
    Document,
    HybridRRFRetriever,
    tokenize_mixed_cn_en,
)


class BGERerankerSimulator:
    """
    类 BAAI/bge-reranker-v2-m3 交叉编码精排器（离线可运行版）：
    模拟 Cross-Encoder 将 (Query, Doc) 拼接后的细粒度交互特征：
    1. 查询关键实体/错误码在文档中的精确覆盖率；
    2. 连续二元词组（Bigram）匹配密度；
    3. 章节标题（section）强相关加权；
    4. 通过 Sigmoid 映射为 [0, 1] 规范化相关概率分数。
    """

    def score_pair(self, query: str, doc: Document) -> float:
        q_tokens = set(tokenize_mixed_cn_en(query))
        if not q_tokens:
            return 0.0
        d_tokens = set(tokenize_mixed_cn_en(doc.page_content))
        section_tokens = set(tokenize_mixed_cn_en(str(doc.metadata.get("section", ""))))

        # 1. 精确错误码/专有代码匹配（最高权重）
        q_codes = set(re.findall(r"[a-z0-9]+[-_][a-z0-9]+", query.lower()))
        code_hit = 0.0
        if q_codes:
            matched_codes = sum(1 for c in q_codes if c in doc.page_content.lower())
            code_hit = 4.0 * (matched_codes / len(q_codes))

        # 2. 二元词组与全词元覆盖率（模拟词级 Attention 交互）
        q_bigrams = {t for t in q_tokens if len(t) >= 2}
        d_bigrams = {t for t in d_tokens if len(t) >= 2}
        bigram_overlap = len(q_bigrams & d_bigrams) / max(1, len(q_bigrams))
        token_overlap = len(q_tokens & d_tokens) / max(1, len(q_tokens))

        # 3. 标题/章节命中加成
        section_overlap = len(q_bigrams & section_tokens) / max(1, len(q_bigrams)) if q_bigrams else 0.0

        logit = -2.2 + code_hit + 3.8 * bigram_overlap + 1.8 * token_overlap + 1.6 * section_overlap
        prob = 1.0 / (1.0 + math.exp(-logit))
        return round(prob, 4)

    def rerank(
        self,
        query: str,
        candidates: Sequence[Document],
        top_n: int = 3,
    ) -> List[Tuple[Document, float]]:
        scored = [(doc, self.score_pair(query, doc)) for doc in candidates]
        scored.sort(key=lambda x: x[1], reverse=True)
        return scored[:top_n]


class ParentDocumentRetriever:
    """
    父文档检索器（Small-to-Big 策略）：
    - 入库时：将完整父文档存入 docstore，同时切分为小粒度子切片（Child Chunks）写入混合索引；
    - 检索时：先对子切片执行 Hybrid RRF + Reranker 精排，再按 parent_id 去重反查完整父文档。
    """

    def __init__(
        self,
        child_chunk_size: int = 55,
        child_chunk_overlap: int = 12,
        use_rerank: bool = True,
    ) -> None:
        self.child_chunk_size = child_chunk_size
        self.child_chunk_overlap = child_chunk_overlap
        self.use_rerank = use_rerank

        self.parent_store: Dict[str, Document] = {}
        self.bm25 = BM25Index()
        self.dense = DenseVectorIndex()
        self.hybrid = HybridRRFRetriever(self.bm25, self.dense)
        self.reranker = BGERerankerSimulator()
        self.total_child_chunks: int = 0

    def _split_to_children(self, parent_doc: Document, parent_id: str) -> List[Document]:
        text = parent_doc.page_content.strip()
        children: List[Document] = []
        step = max(1, self.child_chunk_size - self.child_chunk_overlap)
        for idx, start in enumerate(range(0, len(text), step)):
            piece = text[start : start + self.child_chunk_size].strip()
            if not piece:
                continue
            children.append(
                Document(
                    page_content=piece,
                    metadata={
                        **parent_doc.metadata,
                        "doc_id": f"{parent_id}#c{idx}",
                        "parent_id": parent_id,
                        "child_index": idx,
                    },
                )
            )
            if start + self.child_chunk_size >= len(text):
                break
        return children

    def add_parent_documents(self, parents: Sequence[Document]) -> int:
        all_children: List[Document] = []
        for idx, p_doc in enumerate(parents):
            parent_id = str(p_doc.metadata.get("doc_id") or f"parent-{idx}")
            self.parent_store[parent_id] = p_doc
            all_children.extend(self._split_to_children(p_doc, parent_id))

        self.bm25.add_documents(all_children)
        self.dense.add_documents(all_children)
        self.total_child_chunks += len(all_children)
        return len(all_children)

    def retrieve_parents(self, query: str, top_k: int = 2) -> List[Tuple[Document, float]]:
        # 第一阶段：Hybrid RRF 对子切片粗排 Top-8
        coarse_hits = self.hybrid.retrieve(query, top_k=8, recall_k=10)
        if not coarse_hits:
            return []

        # 第二阶段：Cross-Encoder 精排子切片
        if self.use_rerank:
            ranked_children = self.reranker.rerank(query, [d for d, _ in coarse_hits], top_n=6)
        else:
            ranked_children = coarse_hits

        # 第三阶段：按 parent_id 去重（保留得分最高的子块分数）并回溯完整父文档
        parent_best: Dict[str, float] = {}
        for child_doc, score in ranked_children:
            pid = str(child_doc.metadata["parent_id"])
            if pid not in parent_best or score > parent_best[pid]:
                parent_best[pid] = score

        sorted_pids = sorted(parent_best.items(), key=lambda x: x[1], reverse=True)[:top_k]
        return [(self.parent_store[pid], score) for pid, score in sorted_pids]


if __name__ == "__main__":
    parent_docs = [
        Document(
            "【星辰智脑运维排障全指南-网关篇】系统统一入口采用 Nginx + Envoy 双层网关。"
            "当监控告警出现错误码 ERR-5042 时，代表网关调用下游大模型推理服务超时（默认 15 秒）。"
            "标准处置流程：第一步登录 Grafana 查看推理节点显存与排队长度；"
            "第二步临时将 proxy_read_timeout 调大至 30 秒；第三步若持续 5 分钟则触发降级熔断并通知值班架构师。",
            {"doc_id": "P-OPS-01", "source": "运维排障全指南.md", "section": "ERR-5042网关超时"},
        ),
        Document(
            "【杭州星辰数科差旅报销总则】员工因公出差须提前 3 天在 OA 发起差旅申请并经直属主管审批。"
            "住宿标准方面，一线城市（北京、上海、深圳、广州）每晚报销上限 600 元，新一线城市每晚 450 元。"
            "餐饮补助按每天 100 元定额随工资发放，无需粘贴餐饮发票；超标住宿部分由个人承担。",
            {"doc_id": "P-FIN-01", "source": "差旅报销总则.md", "section": "差旅住宿与餐补"},
        ),
    ]

    retriever = ParentDocumentRetriever(child_chunk_size=50, child_chunk_overlap=10, use_rerank=True)
    n_children = retriever.add_parent_documents(parent_docs)
    print(f"父文档数: {len(parent_docs)} -> 切分子切片数: {n_children}")

    hits = retriever.retrieve_parents("ERR-5042 报警后第三步怎么处理？", top_k=1)
    top_parent, top_score = hits[0]
    print(f"精排回溯父文档: {top_parent.doc_id} (rerank_score={top_score})")
    print(f"完整父文档长度: {len(top_parent.page_content)} 字 -> 包含完整三步处置流程!")


### 🔍 步骤 2 代码深度解析

1. `BGERerankerSimulator` 清晰还原了 Cross-Encoder 精排的核心优势：把 Query 和 Doc 放在一起计算精确错误码命中、二元词组交互密度与标题权重，并经 Sigmoid 输出规范化概率分。
2. `ParentDocumentRetriever` 在小切片命中 `ERR-5042` 后，回溯返回整篇含「第一步/第二步/第三步」的完整父文档，避免大模型因为切片截断而漏掉第三步熔断指令！

**预期运行输出参考：**
```text
父文档数: 2 -> 切分子切片数: 8
精排回溯父文档: P-OPS-01 (rerank_score=0.9852)
完整父文档长度: 158 字 -> 包含完整三步处置流程!
```


## 🧪 当日自动化单元测试

在终端运行 `pytest versions/day33/tests/test_day33.py -v` 即可验收全部测试断言。下面展示完整测试脚本源码，供对照学习测试写法：

```python
"""Day 33 自动化单元测试：验证 BM25、RRF 混合检索、bge-reranker 精排与父文档检索器。"""

from src.day33_hybrid_rrf import (
    BM25Index,
    DenseVectorIndex,
    Document,
    HybridRRFRetriever,
    build_enterprise_sample_docs,
)
from src.day33_rerank_parent import BGERerankerSimulator, ParentDocumentRetriever


def test_bm25_exact_error_code_and_hybrid_rrf():
    docs = build_enterprise_sample_docs()
    bm25 = BM25Index()
    dense = DenseVectorIndex()
    bm25.add_documents(docs)
    dense.add_documents(docs)
    hybrid = HybridRRFRetriever(bm25, dense, rrf_k=60)

    # 1. 错误码精确查询：BM25 与 Hybrid RRF 均应把 ops-5042 排第 1
    bm25_top1 = bm25.search("ERR-5042 报错怎么排查", k=1)[0][0]
    rrf_top1 = hybrid.retrieve("ERR-5042 报错怎么排查", top_k=1)[0][0]
    assert bm25_top1.doc_id == "ops-5042"
    assert rrf_top1.doc_id == "ops-5042"

    # 2. 内部项目代号查询：应精准命中天枢项目
    proj_top1 = hybrid.retrieve("天枢项目预算找谁审批？", top_k=1)[0][0]
    assert proj_top1.doc_id == "proj-tianshu"

    # 3. 口语语义查询：应精准命中差旅住宿标准
    sem_top1 = hybrid.retrieve("去魔都出差住店报销上限是多少", top_k=1)[0][0]
    assert sem_top1.doc_id == "fin-hotel"


def test_bge_reranker_promotes_exact_answer():
    reranker = BGERerankerSimulator()
    candidates = [
        Document("公司行政部负责园区物业、工位分配与班车管理。", {"section": "行政总览"}),
        Document("错误码 ERR-5042 属于统一网关超时，需优先检查上游推理服务显存并调整超时阈值。", {"section": "ERR-5042排查"}),
        Document("差旅报销需要上传电子发票到财务系统。", {"section": "财务流程"}),
    ]
    ranked = reranker.rerank("ERR-5042 网关超时排查方法", candidates, top_n=2)
    assert "ERR-5042" in ranked[0][0].page_content
    assert ranked[0][1] > ranked[1][1]
    assert 0.0 <= ranked[0][1] <= 1.0


def test_parent_document_retriever_deduplicates_and_returns_full_parent():
    p1 = Document(
        "【网关排障】遇到 ERR-5042 超时故障时：第一步查 Grafana 显存；"
        "第二步调大 proxy_read_timeout；第三步开启降级熔断并通知值班架构师。",
        {"doc_id": "P1", "section": "ERR-5042"},
    )
    p2 = Document(
        "【休假规范】入职满 1 年享 5 天年假，病假超 2 天须附公立医院诊断证明。",
        {"doc_id": "P2", "section": "休假规定"},
    )
    retriever = ParentDocumentRetriever(child_chunk_size=30, child_chunk_overlap=8, use_rerank=True)
    child_count = retriever.add_parent_documents([p1, p2])
    assert child_count >= 4

    results = retriever.retrieve_parents("ERR-5042 故障第三步怎么处置？", top_k=2)
    assert len(results) >= 1
    top_doc, score = results[0]
    assert top_doc.doc_id == "P1"
    assert "第一步" in top_doc.page_content and "第三步开启降级熔断" in top_doc.page_content
    # 确保 parent_id 无重复
    returned_ids = [d.doc_id for d, _ in results]
    assert len(returned_ids) == len(set(returned_ids))

```


## 📝 课后梯度实战作业与标准答案

建议先在下方新建空白单元格独立完成编码，运行通过后再核对参考答案。


### 作业 1（基础巩固）：作业一：实现自适应动态权重 RRF 融合器（Query 路由调权）

**题目要求：**
当用户 Query 包含错误码、工单号或英文缩写（如 `ERR-5042`、`VPN`）时，自动将 RRF 权重调整为 `bm25_weight=0.75, dense_weight=0.25`；当 Query 为纯中文口语时调整为 `bm25_weight=0.35, dense_weight=0.65`。请实现 `adaptive_rrf_weights(query)` 并验证。

**思路提示：** 用正则检测 `[A-Za-z0-9]+[-_][A-Za-z0-9]+` 或全大写缩写词来决定权重分配。


In [ ]:
"""Day 33 作业一参考答案：自适应动态权重 RRF 路由器"""
import re
from typing import Tuple


def adaptive_rrf_weights(query: str) -> Tuple[float, float]:
    """返回 (bm25_weight, dense_weight)。"""
    has_code = bool(re.search(r"[A-Za-z]{2,}[-_][0-9]+|[A-Z]{3,}", query))
    if has_code:
        return (0.75, 0.25)
    return (0.35, 0.65)


if __name__ == "__main__":
    w1 = adaptive_rrf_weights("ERR-5042 网关超时咋处理")
    w2 = adaptive_rrf_weights("去魔都出差住店能报多少")
    print("错误码查询权重:", w1)
    print("口语语义查询权重:", w2)
    assert w1 == (0.75, 0.25) and w2 == (0.35, 0.65)


**答案解析与评分标准：**

静态 0.5/0.5 只是起点，根据 Query 形态动态调节 BM25 与向量权重是搜索大厂常见的精调手段。


### 作业 2（企业实战）：作业二：实现 Sentence-Window（句子窗口扩展）检索器

**题目要求：**
将一篇文档按句号拆分为句子列表，索引每个单句；检索命中第 `i` 句后，自动拼接前后各 `window=1` 句（即 `[i-1, i, i+1]`）作为最终上下文返回。请实现 `SentenceWindowRetriever` 并验证。

**思路提示：** 在每条句子的 `metadata` 中记录 `doc_id` 和 `sent_idx`，命中小句后按 `max(0, idx-window) : idx+window+1` 拼接原文。


In [ ]:
"""Day 33 作业二参考答案：Sentence-Window 句子窗口扩展检索器"""
import re
from typing import Dict, List


class SentenceWindowRetriever:
    def __init__(self, window: int = 1) -> None:
        self.window = window
        self.doc_sentences: Dict[str, List[str]] = {}

    def index_document(self, doc_id: str, text: str) -> None:
        sents = [s.strip() for s in re.split(r"(?<=[。！？])", text) if s.strip()]
        self.doc_sentences[doc_id] = sents

    def search_with_window(self, keyword: str) -> List[str]:
        outputs: List[str] = []
        for doc_id, sents in self.doc_sentences.items():
            for idx, s in enumerate(sents):
                if keyword in s:
                    left = max(0, idx - self.window)
                    right = min(len(sents), idx + self.window + 1)
                    outputs.append("".join(sents[left:right]))
        return outputs


if __name__ == "__main__":
    sw = SentenceWindowRetriever(window=1)
    sw.index_document(
        "D1",
        "网关默认超时时间为15秒。遇到ERR-5042需优先排查显存水位。若持续超5分钟请立即开启熔断。",
    )
    res = sw.search_with_window("ERR-5042")
    print("窗口扩展结果:", res[0])
    assert "15秒" in res[0] and "熔断" in res[0]


**答案解析与评分标准：**

Sentence-Window 是 ParentDocumentRetriever 的轻量级兄弟，在长技术文档问答中兼顾精准命中与前后文连贯性。


### 作业 3（架构进阶）：作业三：实现带延迟统计的 Top-K 精排截断阈值滤波器

**题目要求：**
对 Reranker 输出的 `(doc, score)` 列表，增加动态截断逻辑：过滤掉 `score < min_score`（如 `0.45`）的低分噪音文档，同时如果第 1 名与第 2 名分差超过 `0.35`（断层领先），则只返回 Top-1 避免次优文档干扰大模型。请实现 `smart_rerank_cutoff` 并测试两种分支。

**思路提示：** 先按 `min_score` 过滤，再判断 `len(filtered) >= 2 and filtered[0][1] - filtered[1][1] > gap_threshold`。


In [ ]:
"""Day 33 作业三参考答案：精排动态截断阈值滤波器"""
from typing import List, Tuple


def smart_rerank_cutoff(
    ranked_items: List[Tuple[str, float]],
    min_score: float = 0.45,
    gap_threshold: float = 0.35,
) -> List[Tuple[str, float]]:
    filtered = [(doc, sc) for doc, sc in ranked_items if sc >= min_score]
    if len(filtered) >= 2 and (filtered[0][1] - filtered[1][1]) > gap_threshold:
        return [filtered[0]]
    return filtered


if __name__ == "__main__":
    case_gap = [("最优文档", 0.95), ("弱相关文档", 0.50), ("噪音文档", 0.20)]
    case_close = [("文档A", 0.88), ("文档B", 0.82), ("噪音文档", 0.15)]
    out1 = smart_rerank_cutoff(case_gap)
    out2 = smart_rerank_cutoff(case_close)
    print("断层领先截断:", out1)
    print("势均力敌保留:", out2)
    assert len(out1) == 1 and len(out2) == 2


**答案解析与评分标准：**

固定返回 Top-3 往往会在「只需要 1 条证据」时硬塞 2 条噪音；动态分差截断能显著降低 LLM 幻觉率与输入 Token 成本。


---

## 🎯 老陈收尾复盘

漂亮！今天我们把「BM25 + 向量 + RRF + Reranker + 父文档检索」五件套全部手写跑通了。以后面试官问你「生产级 RAG 检索链路怎么设计」，你直接把今天的「多路粗排 -> RRF 融合 -> Cross-Encoder 精排 -> 父文档回溯」四级漏斗画出来，稳拿 Offer！明天咱们做 RAG 自动化评估体系，用数字给今天的新引擎打分。
